## Historic SUMMA ensemble

Each member keeps its own forcing. SUMMA runs once from start to end, then mizuRoute routes that member. No assimilation and no ESP.

In [ ]:
from pathlib import Path

%load_ext autoreload
%autoreload 2

from snowda.config.loader import load_config
from snowda.workflows.run_summa_historic import run_summa_historic

config_path = Path("../runs/examples/ross_summa_historic_ensemble.yaml").resolve()
config = load_config(config_path)
run_dir = run_summa_historic(config)
print("Historic run dir:", run_dir)

## Historic ensemble

Gray lines are every historic ensemble member. There are no assimilation times and no ESP forecasts. Streamflow is mizuRoute discharge for the full window. The orange line is the gauge.

In [ ]:
from pathlib import Path

import yaml

from snowda.io.da_traces import load_da_streamflow, load_da_swe
from snowda.io.streamflow import read_streamflow_observations
from snowda.viz.summa_da import plot_da_traces

catchment = "chena"
if "run_dir" in globals() and (Path(run_dir) / "summa").is_dir():
    historic_run_dir = Path(run_dir)
else:
    run_root = Path(f"../outputs/{catchment}_summa_historic")
    historic_run_dir = max(
        (path for path in run_root.iterdir() if (path / "summa").is_dir()),
        key=lambda path: path.stat().st_mtime,
    )
print("Plotting", historic_run_dir)

figure_dir = historic_run_dir / "figures"
figure_dir.mkdir(exist_ok=True)

swe = load_da_swe(historic_run_dir)
fig, _ax = plot_da_traces(
    swe,
    title=f"{catchment} SWE: historic ensemble",
)
fig.savefig(figure_dir / "historic_swe.png", dpi=150, bbox_inches="tight")

eval_cfg = yaml.safe_load(
    Path(f"../runs/examples/{catchment}_evaluate_streamflow.yaml").read_text()
)
obs = read_streamflow_observations(eval_cfg["obs_csv"])
topology_nc = historic_run_dir / "mizuroute" / "topology.nc"
if not topology_nc.is_file():
    det_root = Path(f"../outputs/{catchment}_summa_deterministic")
    candidates = sorted(
        det_root.glob("*/mizuroute/topology.nc"),
        key=lambda path: path.stat().st_mtime,
    )
    topology_nc = candidates[-1] if candidates else None
flow = load_da_streamflow(historic_run_dir, topology_nc=topology_nc)
fig, _ax = plot_da_traces(
    flow,
    obs,
    title=f"{catchment} streamflow: historic ensemble",
)
fig.savefig(figure_dir / "historic_streamflow.png", dpi=150, bbox_inches="tight")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from snowda.obs_qm.seasonal import water_year


def _trace_times(traces) -> pd.DatetimeIndex:
    pieces = [
        pd.DatetimeIndex(pd.to_datetime(segment["time"].values))
        for member in traces.historic
        for segment in member.segments
    ]
    if not pieces:
        return pd.DatetimeIndex([])
    times = pieces[0]
    for piece in pieces[1:]:
        times = times.union(piece)
    return times


# Water year starts 1 October: 1 Oct 2015 belongs to water year 2016.
times = _trace_times(swe).union(_trace_times(flow))
water_years = sorted({int(year) for year in water_year(times)})
wy_dir = figure_dir / "water_years"
wy_dir.mkdir(exist_ok=True)

for wy in water_years:
    start = pd.Timestamp(year=wy - 1, month=10, day=1)
    end = pd.Timestamp(year=wy, month=10, day=1) - pd.Timedelta(seconds=1)
    fig, _ax = plot_da_traces(
        swe,
        start=start,
        end=end,
        title=f"Ross SWE: water year {wy}",
    )
    fig.savefig(wy_dir / f"historic_swe_wy{wy}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    fig, _ax = plot_da_traces(
        flow,
        obs,
        start=start,
        end=end,
        title=f"Ross streamflow: water year {wy}",
    )
    fig.savefig(wy_dir / f"historic_streamflow_wy{wy}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Wrote {len(water_years)} water years to {wy_dir}")
